# 🧩 Split Pose Dataset — BẢN SẠCH (chống data leak)
**Nguồn:** `KLTN_AGAIN_4_DIEM.v2-test_211_anh.coco-segmentation` (211 ảnh, 4 lớp)

Quy trình (khác bản cũ V3 để **loại data leak**):
1. **Convert** COCO (polygon `segmentation` hoặc `keypoints`) → 4 keypoint, **giữ nguyên thứ tự đã click**, chỉ chuẩn hoá chiều duyệt.
2. **CHIA TRƯỚC** 80/10/10 theo ảnh gốc (seed 42).
3. **AUGMENT CHỈ TRÊN TRAIN** (keypoint-safe: chỉ đổi màu/nhiễu). Val/Test giữ nguyên ảnh gốc.
4. Tạo `dataset.yaml` (`kpt_shape [4,3]`, `flip_idx [1,0,3,2]`, 4 lớp).
5. **Cell kiểm tra leak** (bắt buộc = 0) + vẽ keypoint kiểm tra thứ tự.
6. Nén `.zip` để Add Input lên Kaggle.

> ⚠️ Điểm mấu chốt: **CHIA trước, AUGMENT sau — và chỉ augment train**. Bản cũ augment *toàn bộ* ảnh rồi mới chia → bản augment của ảnh val/test lọt vào train (leak).

## 📦 Cell 1: Cấu hình & Import

In [1]:
import os, json, glob, shutil, random, io, collections
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

# ===================== CẤU HÌNH =====================
COCO_DIR   = r"E:\AZIMUTH_TEST_LAN1\KLTN 2026.v2-kltn_data_pose_chua_tang_cuong.coco-segmentation\train"
COCO_JSON  = os.path.join(COCO_DIR, "_annotations.coco.json")
OUTPUT_DIR = r"E:\AZIMUTH_TEST_LAN1\KLTN_POSE_FINAL_2026"

RATIO = (0.8, 0.1, 0.1)      # train / val / test theo ẢNH GỐC
SEED  = 42
NUM_KP = 4
CATEGORY_MAP = {1: 0, 2: 1, 3: 2, 4: 3}          # COCO cat_id -> YOLO class_id
CLASS_NAMES  = ["anten-4G", "anten-5G", "rrh", "rru"]
FLIP_IDX     = [1, 0, 3, 2]                        # lật ngang: TL<->TR, BR<->BL

DO_AUGMENT    = True          # augment CHỈ trên train (keypoint-safe)
AUG_PER_IMAGE = 3             # số biến thể sinh thêm cho mỗi ảnh train

random.seed(SEED); np.random.seed(SEED)
print("[OK] Config | cv2:", HAS_CV2)
print("  Nguồn :", COCO_DIR)
print("  Xuất  :", OUTPUT_DIR)

[OK] Config | cv2: True
  Nguồn : E:\AZIMUTH_TEST_LAN1\archive\old_datasets\KLTN_AGAIN_4_DIEM.v2-test_211_anh.coco-segmentation\train
  Xuất  : E:\AZIMUTH_TEST_LAN1\KLTN_POSE_V6_CLEAN


## 📂 Cell 2: Nạp COCO & thống kê

In [2]:
coco = json.load(open(COCO_JSON, encoding="utf-8"))
imgs_info  = {im["id"]: im for im in coco["images"]}
ann_by_img = {}
for a in coco["annotations"]:
    ann_by_img.setdefault(a["image_id"], []).append(a)

id2 = {c["id"]: c["name"] for c in coco["categories"]}
cc  = collections.Counter(a["category_id"] for a in coco["annotations"])
print("Ảnh:", len(coco["images"]), "| Annotation:", len(coco["annotations"]))
print("Phân bố lớp:", {id2[k]: v for k, v in sorted(cc.items()) if k in CATEGORY_MAP})

Ảnh: 211 | Annotation: 565
Phân bố lớp: {'anten-4G': 391, 'anten-5G': 55, 'rrh': 35, 'rru': 84}


## 🔄 Cell 3: Convert polygon → 4 keypoint (GIỮ NGUYÊN thứ tự đã click)
Đọc thẳng thứ tự 4 điểm trong file COCO, **chỉ chuẩn hoá chiều duyệt**.
KHÔNG sắp lại bằng heuristic toạ độ — đã đo và chứng minh mọi cách sắp lại đều làm hỏng nhãn (chi tiết trong chú thích ở cell dưới).

In [3]:
# ĐỌC THẲNG thứ tự 4 điểm đã click trong file COCO — KHÔNG sắp lại.
#
# Vì sao bỏ order_4_corners: đo trên 568 annotation nguồn thì nhãn gốc vốn đã sạch —
#   tứ giác tự cắt 0/568 · điểm trùng 0/568 · cùng chiều duyệt 567/568 (99.8%)
#   · nhìn ảnh đúng TL → TR → BR → BL.
# Mọi heuristic 2D để "sắp lại" đều gãy khi anten bị xoay + phối cảnh, và chỉ làm hỏng nhãn:
#   - tổng/hiệu toạ độ : dồn 4 góc thành 2 điểm ở 22.4% nhãn (tứ giác "sập")
#   - cạnh trên cùng   : xoay lệch 1 bậc ở 8.8% nhãn (rrh 64%, anten-5G 22%)
#     -> model học nhãn trộn quy ước -> solvePnP lắp hình xoay 90° -> tilt ra số âm phi lý.
# Con người nhìn được ảnh nên biết góc nào thật sự là trên-trái kể cả khi anten xiên; script thì không.
# Thứ duy nhất thật sự tuỳ tiện là CHIỀU DUYỆT -> chỉ chuẩn hoá đúng cái đó.

def dien_tich_co_dau(p):
    """Shoelace. Dấu = chiều duyệt (bất biến với phép xoay và tịnh tiến)."""
    x, y = p[:, 0], p[:, 1]
    return 0.5 * (np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def do_xeo(p):
    """2 cạnh đối của hình chữ nhật thật phải bằng nhau; chiếu càng xiên càng lệch.
    Lệch quá ngưỡng => polygon vẽ sai, không phải do phối cảnh."""
    e = [np.linalg.norm(p[i] - p[(i + 1) % 4]) for i in range(4)]
    return max(max(e[0], e[2]) / max(1e-9, min(e[0], e[2])),
               max(e[1], e[3]) / max(1e-9, min(e[1], e[3])))

def lay_4_diem(a):
    """Ưu tiên polygon 'segmentation'; không có thì lấy 'keypoints'
    (bản export COCO Keypoints để segmentation rỗng -> nếu chỉ đọc segmentation sẽ ra 0 nhãn)."""
    seg = a.get("segmentation") or []
    if seg and isinstance(seg[0], list) and len(seg[0]) >= 8:
        return np.asarray(seg[0][:8], dtype=np.float64).reshape(-1, 2)
    kp = a.get("keypoints") or []
    if len(kp) >= 12:
        return np.asarray([[kp[i], kp[i + 1]] for i in range(0, 12, 3)], dtype=np.float64)
    return None

def clamp01(v): return max(0.0, min(1.0, v))

THONG_KE = collections.Counter()

def convert_image(img_id):
    im = imgs_info[img_id]; W, H = im["width"], im["height"]
    lines = []
    for a in ann_by_img.get(img_id, []):
        cid = a["category_id"]
        if cid not in CATEGORY_MAP:
            continue
        p = lay_4_diem(a)
        if p is None:
            THONG_KE["bỏ: không có polygon lẫn keypoint"] += 1; continue
        if len({(round(x, 3), round(y, 3)) for x, y in p}) != 4:
            THONG_KE["bỏ: 4 điểm không phân biệt"] += 1; continue
        if do_xeo(p) > 2.0:
            THONG_KE["bỏ: polygon vẽ sai (độ xéo > 2)"] += 1; continue
        if dien_tich_co_dau(p) < 0:          # đi ngược số đông -> đảo chiều, GIỮ NGUYÊN P0
            p = p[[0, 3, 2, 1]]
            THONG_KE["đã đảo chiều duyệt"] += 1
        else:
            THONG_KE["giữ nguyên thứ tự click"] += 1

        bx, by, bw, bh = a["bbox"]
        vals = [clamp01((bx + bw / 2) / W), clamp01((by + bh / 2) / H),
                clamp01(bw / W), clamp01(bh / H)]
        for (x, y) in p:
            vals += [clamp01(x / W), clamp01(y / H), 2]      # v=2: visible
        lines.append(str(CATEGORY_MAP[cid]) + " " + " ".join(f"{v:.6f}" for v in vals))
    return im["file_name"], lines

converted = {}
for img_id in imgs_info:
    fn, lines = convert_image(img_id)
    if lines and os.path.exists(os.path.join(COCO_DIR, fn)):
        converted[fn] = lines

print("Ảnh có ≥1 nhãn hợp lệ:", len(converted))
for k, v in THONG_KE.most_common():
    print(f"   {k:34} {v}")
assert converted, "KHÔNG có nhãn nào -> kiểm tra định dạng export (segmentation / keypoints)"


Ảnh có ≥1 nhãn hợp lệ: 210


## ✂️ Cell 4: CHIA TRƯỚC 80/10/10 (theo ảnh gốc)
Chia trên **ảnh gốc**, val/test chỉ gồm ảnh gốc — nền tảng để không leak.

In [4]:
files = sorted(converted.keys())
random.seed(SEED); random.shuffle(files)          # tái lập được
n = len(files)
n_val  = max(1, round(n * RATIO[1]))
n_test = max(1, round(n * RATIO[2]))
val_files   = files[:n_val]
test_files  = files[n_val:n_val + n_test]
train_files = files[n_val + n_test:]
print(f"CHIA GỐC (trước augment): train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for sp in ("train", "val", "test"):
    os.makedirs(os.path.join(OUTPUT_DIR, sp, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, sp, "labels"), exist_ok=True)

def write_split(split, flist):
    for fn in flist:
        base = os.path.splitext(fn)[0]
        shutil.copy2(os.path.join(COCO_DIR, fn),
                     os.path.join(OUTPUT_DIR, split, "images", fn))
        with open(os.path.join(OUTPUT_DIR, split, "labels", base + ".txt"), "w") as f:
            f.write("\n".join(converted[fn]))

write_split("train", train_files)
write_split("val",   val_files)
write_split("test",  test_files)
print("[OK] Đã ghi ảnh GỐC + nhãn (val/test giữ nguyên, KHÔNG augment).")

CHIA GỐC (trước augment): train=168  val=21  test=21
[OK] Đã ghi ảnh GỐC + nhãn (val/test giữ nguyên, KHÔNG augment).


## 🎨 Cell 5: AUGMENT CHỈ TRÊN TRAIN (keypoint-safe)
Chỉ đổi màu/nhiễu (không biến đổi hình học) → keypoint giữ nguyên → **nhãn copy y nguyên**. Val/Test không đụng tới.

In [5]:
AUG_TYPES = ["brightness_up","brightness_down","contrast_up","contrast_down",
             "saturation_up","saturation_down","gaussian_noise","blur","sharpen",
             "jpeg_compress","channel_shuffle","grayscale_mix"]

def aug_one(img, t):
    if t=="brightness_up":   return ImageEnhance.Brightness(img).enhance(random.uniform(1.15,1.45))
    if t=="brightness_down": return ImageEnhance.Brightness(img).enhance(random.uniform(0.55,0.85))
    if t=="contrast_up":     return ImageEnhance.Contrast(img).enhance(random.uniform(1.2,1.6))
    if t=="contrast_down":   return ImageEnhance.Contrast(img).enhance(random.uniform(0.5,0.8))
    if t=="saturation_up":   return ImageEnhance.Color(img).enhance(random.uniform(1.3,1.8))
    if t=="saturation_down": return ImageEnhance.Color(img).enhance(random.uniform(0.3,0.7))
    if t=="gaussian_noise":
        a=np.array(img).astype(np.int16); nz=np.random.normal(0,random.uniform(8,25),a.shape).astype(np.int16)
        return Image.fromarray(np.clip(a+nz,0,255).astype(np.uint8))
    if t=="blur":    return img.filter(ImageFilter.GaussianBlur(random.uniform(0.5,1.5)))
    if t=="sharpen": return ImageEnhance.Sharpness(img).enhance(random.uniform(1.5,3.0))
    if t=="jpeg_compress":
        buf=io.BytesIO(); img.save(buf,format="JPEG",quality=random.randint(30,60)); buf.seek(0)
        return Image.open(buf).convert("RGB")
    if t=="channel_shuffle":
        ch=list(img.split()); o=[0,1,2]; random.shuffle(o); return Image.merge("RGB",[ch[i] for i in o])
    if t=="grayscale_mix":
        a=np.array(img).astype(np.float32); g=np.array(img.convert("L").convert("RGB")).astype(np.float32)
        al=random.uniform(0.3,0.7); return Image.fromarray((al*a+(1-al)*g).astype(np.uint8))
    return img

n_aug = 0
if DO_AUGMENT:
    tr_img = os.path.join(OUTPUT_DIR,"train","images")
    tr_lbl = os.path.join(OUTPUT_DIR,"train","labels")
    for fn in list(train_files):                 # CHỈ ảnh train gốc
        base, ext = os.path.splitext(fn)
        try:
            img = Image.open(os.path.join(tr_img, fn)).convert("RGB")
        except Exception:
            continue
        for i in range(AUG_PER_IMAGE):
            t = random.choice(AUG_TYPES)
            out = f"{base}_aug{i}_{t}{ext}"
            aug_one(img, t).save(os.path.join(tr_img, out), quality=95)
            shutil.copy2(os.path.join(tr_lbl, base+".txt"),
                         os.path.join(tr_lbl, f"{base}_aug{i}_{t}.txt"))
            n_aug += 1
print(f"[OK] Augment (chỉ train): +{n_aug} ảnh | train tổng = {len(train_files)+n_aug}")

[OK] Augment (chỉ train): +504 ảnh | train tổng = 672


## 📝 Cell 6: Tạo dataset.yaml

In [6]:
yaml = (
 "# KLTN Pose V6 - CLEAN (split TRUOC, augment CHI train -> khong data leak)\n"
 f"path: {OUTPUT_DIR.replace(os.sep,'/')}\n"
 "train: train/images\nval: val/images\ntest: test/images\n\n"
 f"kpt_shape: [{NUM_KP}, 3]\n"
 f"flip_idx: {FLIP_IDX}\n\n"
 "names:\n" + "".join(f"  {i}: {n}\n" for i,n in enumerate(CLASS_NAMES)) +
 f"nc: {len(CLASS_NAMES)}\n"
)
with open(os.path.join(OUTPUT_DIR,"dataset.yaml"),"w",encoding="utf-8") as f:
    f.write(yaml)
print(yaml)

# KLTN Pose V6 - CLEAN (split TRUOC, augment CHI train -> khong data leak)
path: E:/AZIMUTH_TEST_LAN1/KLTN_POSE_V6_CLEAN
train: train/images
val: val/images
test: test/images

kpt_shape: [4, 3]
flip_idx: [1, 0, 3, 2]

names:
  0: anten-4G
  1: anten-5G
  2: rrh
  3: rru
nc: 4



## ✅ Cell 7: KIỂM TRA DATA LEAK + phân bố lớp
Base-name gốc của train/val/test phải rời nhau hoàn toàn (= 0).

In [7]:
def root_base(fn):  # bỏ hậu tố _aug... để lấy tên ảnh gốc
    return os.path.splitext(fn)[0].split("_aug")[0]

def imgs(sp): return os.listdir(os.path.join(OUTPUT_DIR, sp, "images"))
r_tr = {root_base(f) for f in imgs("train")}
r_va = {root_base(f) for f in imgs("val")}
r_te = {root_base(f) for f in imgs("test")}
print("=== KIỂM TRA LEAK (phải = 0 hết) ===")
print("  train giao val :", len(r_tr & r_va))
print("  train giao test:", len(r_tr & r_te))
print("  val   giao test:", len(r_va & r_te))
assert r_tr.isdisjoint(r_va) and r_tr.isdisjoint(r_te) and r_va.isdisjoint(r_te), "❌ CÓ LEAK!"
print("  => KHÔNG LEAK ✔")

EXP = 1 + 4 + NUM_KP*3        # 17 trường/dòng
print("\n=== Phân bố lớp & format nhãn ===")
for sp in ("train","val","test"):
    lbls = glob.glob(os.path.join(OUTPUT_DIR, sp, "labels", "*.txt"))
    cd = collections.Counter(); bad = 0; obj = 0
    for lf in lbls:
        for ln in open(lf):
            ln = ln.strip()
            if not ln: continue
            p = ln.split()
            if len(p) != EXP: bad += 1; continue
            cd[CLASS_NAMES[int(p[0])]] += 1; obj += 1
    print(f"[{sp:5}] {len(lbls):4} ảnh | {obj:5} object | sai format: {bad} | {dict(cd)}")

=== KIỂM TRA LEAK (phải = 0 hết) ===
  train giao val : 0
  train giao test: 0
  val   giao test: 0
  => KHÔNG LEAK ✔

=== Phân bố lớp & format nhãn ===
[train]  672 ảnh |  1820 object | sai format: 0 | {'anten-5G': 172, 'rru': 264, 'anten-4G': 1260, 'rrh': 124}
[val  ]   21 ảnh |    58 object | sai format: 0 | {'anten-4G': 40, 'rru': 10, 'anten-5G': 6, 'rrh': 2}
[test ]   21 ảnh |    52 object | sai format: 0 | {'anten-4G': 36, 'rrh': 2, 'rru': 8, 'anten-5G': 6}


## 👁️ Cell 8 (tùy chọn): Vẽ keypoint kiểm tra thứ tự
0=TL(đỏ) · 1=TR(lục) · 2=BR(lam) · 3=BL(vàng). Nhìn vài ảnh xem 4 góc có đúng vị trí không.

In [8]:
if HAS_CV2:
    prev = os.path.join(OUTPUT_DIR, "_preview_keypoints"); os.makedirs(prev, exist_ok=True)
    COLORS = [(0,0,255),(0,255,0),(255,0,0),(0,255,255)]   # BGR: TL đỏ, TR lục, BR lam, BL vàng
    sample = [f for f in os.listdir(os.path.join(OUTPUT_DIR,"train","images")) if "_aug" not in f][:8]
    for fn in sample:
        img = cv2.imread(os.path.join(OUTPUT_DIR,"train","images",fn)); h,w = img.shape[:2]
        base = os.path.splitext(fn)[0]
        for ln in open(os.path.join(OUTPUT_DIR,"train","labels",base+".txt")):
            p = ln.split()
            if len(p) != 1+4+NUM_KP*3: continue
            for k in range(NUM_KP):
                x = int(float(p[5+k*3])*w); y = int(float(p[5+k*3+1])*h)
                cv2.circle(img,(x,y),4,COLORS[k],-1)
                cv2.putText(img,str(k),(x+3,y-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,COLORS[k],1)
        cv2.imwrite(os.path.join(prev,fn), img)
    print("Đã lưu ảnh preview ->", prev)
else:
    print("Không có cv2 -> bỏ qua preview.")

Đã lưu ảnh preview -> E:\AZIMUTH_TEST_LAN1\KLTN_POSE_V6_CLEAN\_preview_keypoints


## 📦 Cell 9: Nén .zip để Add Input lên Kaggle

In [9]:
zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
sz = os.path.getsize(zip_path) / (1024*1024)
print(f"[OK] Đã nén: {zip_path}  ({sz:.1f} MB)")
print("=> Add Input file .zip này lên Kaggle, rồi train bằng yolo26-pose.ipynb.")

[OK] Đã nén: E:\AZIMUTH_TEST_LAN1\KLTN_POSE_V6_CLEAN.zip  (376.6 MB)
=> Add Input file .zip này lên Kaggle, rồi train bằng yolo26-pose.ipynb.
